# SupportOps AI - Baseline Ticket Classification

## Objective

Build a baseline NLP classification model that predicts the type of a
customer support ticket from its text.

### Pipeline

Ticket Text → TF-IDF → Logistic Regression → Ticket Type

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully!")

In [ ]:
DATA_PATH = "../data/processed/ticket_classification_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Verify the data before training

In [ ]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTicket classes:")
print(df["ticket_type"].value_counts())

In [ ]:
X = df["ticket_text"]
y = df["ticket_type"]

In [ ]:
print("Input examples:", X.shape[0])
print("Target examples:", y.shape[0])

Split the dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Verify stratification

In [ ]:
train_distribution = (
    y_train.value_counts(normalize=True) * 100
)

test_distribution = (
    y_test.value_counts(normalize=True) * 100
)

distribution_comparison = pd.DataFrame({
    "Train %": train_distribution,
    "Test %": test_distribution
})

distribution_comparison

Create the TF-IDF vectorizer

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

Build the Pipeline

In [ ]:
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

Train the model

In [ ]:
baseline_model.fit(
    X_train,
    y_train
)

print("Baseline model trained successfully!")

Generate Predictions

In [ ]:
y_pred = baseline_model.predict(X_test)

print("Predictions generated!")

Calculating accuracy score

In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"Accuracy: {accuracy:.4f}")

Precision, Recall and F1

In [ ]:
precision = precision_score(
    y_test,
    y_pred,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Macro F1:  {f1:.4f}")

Generating full classification report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)

Creating the confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=45,
    ax=ax
)

plt.title("Baseline Model - Confusion Matrix")
plt.tight_layout()
plt.show()

Creating a compact metrics table

In [ ]:
metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

metrics_df

Inspect actual predictions

In [ ]:
results_df = pd.DataFrame({
    "ticket_text": X_test,
    "actual": y_test,
    "predicted": y_pred
})

results_df.head(10)

In [ ]:
errors_df = results_df[
    results_df["actual"] != results_df["predicted"]
]

print("Number of incorrect predictions:", len(errors_df))

errors_df.head(20)

Inspect individual mistakes

In [ ]:
for _, row in errors_df.head(5).iterrows():

    print("=" * 80)

    print("TICKET:")
    print(row["ticket_text"])

    print("\nACTUAL:")
    print(row["actual"])

    print("\nPREDICTED:")
    print(row["predicted"])

    print()

In [ ]:
train_texts = set(X_train)
test_texts = set(X_test)

overlap = train_texts.intersection(test_texts)

print(
    "Exact ticket texts appearing in both train and test:",
    len(overlap)
)

29 exact ticket texts appear in both your training set and your test set. That means our current evaluation has a small amount of train–test leakage, so we should fix it before trusting the baseline metrics.

In [ ]:
overlap_texts = train_texts.intersection(test_texts)

print("Number of overlapping texts:", len(overlap_texts))

for text in list(overlap_texts)[:10]:
    print("=" * 80)
    print(text)

Check whether identical text has the same or different label

In [ ]:
duplicate_label_check = (
    df.groupby("ticket_text")["ticket_type"]
      .nunique()
)

conflicting_texts = duplicate_label_check[
    duplicate_label_check > 1
]

print(
    "Exact same texts having different labels:",
    len(conflicting_texts)
)

Clean the conflicts properly

In [ ]:
conflicting_text_values = set(
    conflicting_texts.index
)

clean_model_df = df[
    ~df["ticket_text"].isin(
        conflicting_text_values
    )
].copy()

print("Before conflict removal:", df.shape)
print("After conflict removal:", clean_model_df.shape)

In [ ]:
clean_model_df = clean_model_df.drop_duplicates(
    subset=["ticket_text"]
).copy()

print(
    "After duplicate removal:",
    clean_model_df.shape
)

In [ ]:
print(
    "Duplicate texts remaining:",
    clean_model_df.duplicated(
        subset=["ticket_text"]
    ).sum()
)

verify that there are no remaining conflicting labels

In [ ]:
remaining_conflicts = (
    clean_model_df
    .groupby("ticket_text")["ticket_type"]
    .nunique()
)

remaining_conflicts = remaining_conflicts[
    remaining_conflicts > 1
]

print(
    "Remaining conflicting texts:",
    len(remaining_conflicts)
)

recreate X and y

In [ ]:
X = clean_model_df["ticket_text"]
y = clean_model_df["ticket_type"]

Split again

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Checking overlap again

In [ ]:
train_texts = set(X_train)
test_texts = set(X_test)

overlap = train_texts.intersection(test_texts)

print(
    "Exact ticket texts appearing in both train and test:",
    len(overlap)
)

Retrain the corrected baseline

In [ ]:
baseline_model.fit(
    X_train,
    y_train
)

print("Baseline model retrained successfully!")

generate predictions

In [ ]:
y_pred = baseline_model.predict(X_test)

print("Predictions generated!")

calculate the corrected metrics

In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall:    {recall:.4f}")
print(f"Macro F1:        {f1:.4f}")

detailed report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)

new confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=45,
    ax=ax
)

plt.title(
    "TF-IDF + Logistic Regression\n"
    "Confusion Matrix After Data Quality Fixes"
)

plt.tight_layout()
plt.show()

recreate error analysis

In [ ]:
results_df = pd.DataFrame({
    "ticket_text": X_test,
    "actual": y_test,
    "predicted": y_pred
})

errors_df = results_df[
    results_df["actual"]
    != results_df["predicted"]
].copy()

print(
    "Correct predictions:",
    len(results_df) - len(errors_df)
)

print(
    "Incorrect predictions:",
    len(errors_df)
)

In [ ]:
for _, row in errors_df.head(10).iterrows():

    print("=" * 80)

    print("TICKET:")
    print(row["ticket_text"])

    print("\nACTUAL:")
    print(row["actual"])

    print("\nPREDICTED:")
    print(row["predicted"])

    print()

In [ ]:
def get_prediction_details(text):

    probabilities = baseline_model.predict_proba(
        [text]
    )[0]

    classes = baseline_model.classes_

    probability_df = pd.DataFrame({
        "ticket_type": classes,
        "probability": probabilities
    })

    return probability_df.sort_values(
        "probability",
        ascending=False
    )

In [ ]:
sample_error = errors_df.iloc[0]

print("Ticket:")
print(sample_error["ticket_text"])

print("\nActual:")
print(sample_error["actual"])

print("\nPredicted:")
print(sample_error["predicted"])

get_prediction_details(
    sample_error["ticket_text"]
)

In [ ]:
full_df = pd.read_csv(
    "../data/processed/customer_support_tickets_clean.csv"
)

In [ ]:
pd.crosstab(
    full_df["ticket_subject"],
    full_df["ticket_type"]
)

In [ ]:
subjects_to_check = [
    "Refund request",
    "Cancellation request",
    "Installation support",
    "Product setup",
    "Account access"
]

for subject in subjects_to_check:
    
    if subject in full_df["ticket_subject"].values:
        
        print("\n", "=" * 60)
        print("SUBJECT:", subject)
        print("=" * 60)
        
        print(
            full_df[
                full_df["ticket_subject"] == subject
            ]["ticket_type"].value_counts()
        )